<a href="https://colab.research.google.com/github/davwolfe/davwolfe/blob/main/CS617_Assignment3_BostonCrime.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
%pylab inline

Populating the interactive namespace from numpy and matplotlib


In [31]:
sql = '''
SELECT "HOUR",
       "DAY_OF_WEEK",
       "OFFENSE_CODE_GROUP",
       "OFFENSE_DESCRIPTION"
FROM "b973d8cb-eeb2-4e7e-99da-c92938efc9c0"
WHERE "HOUR" IS NOT NULL
  AND "DAY_OF_WEEK" IS NOT NULL
'''

In [32]:
bpd_logs = "https://data.boston.gov/api/3/action/datastore_search_sql"

In [33]:
import urllib, requests

In [34]:
url = bpd_logs + "?sql=" + urllib.parse.quote(sql)
rows = requests.get(url).json()

crimes = rows['result']['records']
print(len(crimes), "records fetched")
print(crimes[0])

32000 records fetched
{'HOUR': '22', 'DAY_OF_WEEK': 'Friday   ', 'OFFENSE_CODE_GROUP': None, 'OFFENSE_DESCRIPTION': 'INVESTIGATE PERSON'}


In [35]:
violent_descriptions = [
    'ASSAULT - AGGRAVATED', 'ASSAULT - SIMPLE', 'ROBBERY',
    'THREATS TO DO BODILY HARM', 'HARASSMENT/ CRIMINAL HARASSMENT',
    'MURDER, NON-NEGLIGENT MANSLAUGHTER', 'RAPE', 'SEX OFFENSE',
    'ASSAULT & BATTERY', 'KIDNAPPING', 'HUMAN TRAFFICKING',
    'VERBAL DISPUTE', 'INTIMIDATING WITNESS'
]

violent    = [r for r in crimes if r.get('OFFENSE_DESCRIPTION') in violent_descriptions]
nonviolent = [r for r in crimes if r.get('OFFENSE_DESCRIPTION') not in violent_descriptions
              and r.get('OFFENSE_DESCRIPTION') is not None]

print("Violent:",     len(violent))
print("Non-violent:", len(nonviolent))

Violent: 4994
Non-violent: 27006


In [36]:
from collections import Counter

# aggregate by HOUR
violent_by_hour    = Counter(int(r['HOUR']) for r in violent    if r.get('HOUR') is not None)
nonviolent_by_hour = Counter(int(r['HOUR']) for r in nonviolent if r.get('HOUR') is not None)

hours = list(range(24))
v_hour_counts  = [violent_by_hour.get(h, 0)    for h in hours]
nv_hour_counts = [nonviolent_by_hour.get(h, 0) for h in hours]

print("Violent by hour:",     v_hour_counts)
print("Non-violent by hour:", nv_hour_counts)

Violent by hour: [440, 172, 152, 74, 67, 51, 55, 103, 137, 179, 219, 238, 264, 253, 232, 295, 297, 288, 310, 260, 268, 250, 208, 182]
Non-violent by hour: [2237, 658, 576, 401, 269, 339, 419, 725, 1139, 1332, 1430, 1499, 1727, 1412, 1373, 1341, 1646, 1733, 1520, 1286, 1183, 1145, 911, 705]


In [37]:
# aggregate by DAY_OF_WEEK (.strip() which handles trailing spaces in the data)
day_order = ['Sunday','Monday','Tuesday','Wednesday','Thursday','Friday','Saturday']

violent_by_day    = Counter(r['DAY_OF_WEEK'].strip() for r in violent    if r.get('DAY_OF_WEEK'))
nonviolent_by_day = Counter(r['DAY_OF_WEEK'].strip() for r in nonviolent if r.get('DAY_OF_WEEK'))

v_day_counts  = [violent_by_day.get(d, 0)    for d in day_order]
nv_day_counts = [nonviolent_by_day.get(d, 0) for d in day_order]

print("Violent by day:",     list(zip(day_order, v_day_counts)))
print("Non-violent by day:", list(zip(day_order, nv_day_counts)))

Violent by day: [('Sunday', 761), ('Monday', 721), ('Tuesday', 746), ('Wednesday', 704), ('Thursday', 596), ('Friday', 690), ('Saturday', 776)]
Non-violent by day: [('Sunday', 3399), ('Monday', 3886), ('Tuesday', 4186), ('Wednesday', 3924), ('Thursday', 3697), ('Friday', 3931), ('Saturday', 3983)]


In [38]:
# prints JS variables
print("// Copy these into asst3.html:\n")
print(f"var v_hour  = {v_hour_counts};")
print(f"var nv_hour = {nv_hour_counts};")
print(f"var v_day   = {v_day_counts};")
print(f"var nv_day  = {nv_day_counts};")

// Copy these into asst3.html:

var v_hour  = [440, 172, 152, 74, 67, 51, 55, 103, 137, 179, 219, 238, 264, 253, 232, 295, 297, 288, 310, 260, 268, 250, 208, 182];
var nv_hour = [2237, 658, 576, 401, 269, 339, 419, 725, 1139, 1332, 1430, 1499, 1727, 1412, 1373, 1341, 1646, 1733, 1520, 1286, 1183, 1145, 911, 705];
var v_day   = [761, 721, 746, 704, 596, 690, 776];
var nv_day  = [3399, 3886, 4186, 3924, 3697, 3931, 3983];


In [39]:
from collections import Counter
descs = Counter(r.get('OFFENSE_DESCRIPTION') for r in crimes)
print(descs.most_common(20))

[('INVESTIGATE PERSON', 3218), ('SICK ASSIST', 2745), ('M/V - LEAVING SCENE - PROPERTY DAMAGE', 1779), ('ASSAULT - SIMPLE', 1446), ('TOWED MOTOR VEHICLE', 1403), ('INVESTIGATE PROPERTY', 1338), ('LARCENY SHOPLIFTING', 1237), ('VANDALISM', 1181), ('PROPERTY - LOST/ MISSING', 1030), ('VERBAL DISPUTE', 1002), ('THREATS TO DO BODILY HARM', 817), ('DRUGS - POSSESSION/ SALE/ MANUFACTURING/ USE', 803), ('ASSAULT - AGGRAVATED', 765), ('LARCENY THEFT FROM MV - NON-ACCESSORY', 708), ('LARCENY THEFT FROM BUILDING', 698), ('LARCENY ALL OTHERS', 695), ('M/V ACCIDENT - PROPERTY DAMAGE', 674), ('M/V ACCIDENT - OTHER', 631), ('HARASSMENT/ CRIMINAL HARASSMENT', 580), ('MISSING PERSON - LOCATED', 494)]
